In [44]:
import json
import pickle
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import pairwise_distances
from tqdm import tqdm
import time

In [45]:
def read_raw_data(_num_samples, _fn):
    """
    Reads and processes the raw Goodreads data.
    :param _num_samples: The number of rows to sample from the dataframe.
    :param _fn: Formatted filename of output.
    :return: None, saves the output dataframe.
    """
    _df = pd.read_csv("goodreads_interactions.csv", nrows=_num_samples)
    _df = _df[_df.is_read == 1]
    _df = _df[0:_num_samples]
    _df.to_csv('goodreads_{}.csv'.format(_fn, index=False))


def build_rating_matrix(_df):
    """
    Converts a dataframe to a user-item interaction matrix.
    :param _df: The input dataframe.
    :return: Numpy matrix representing user-interaction ratings.
    """
    _n_users = _df.user_id.max() + 1
    _n_books = _df.book_idx.max() + 1  
    print('Users: {}'.format(_n_users))
    print('Books: {}'.format(_n_books))
    _ratings = np.zeros((_n_users, _n_books))
    for _, row in tqdm(_df.iterrows()):
        i = row.user_id
        j = row.book_idx
        _ratings[i, j] = row.rating
    return _ratings


def recommend_item_similarity(_matrix, _n_latent):
    """
    Builds item similarities using truncated SVD.
    :param _matrix: The user-item rating matrix.
    :param _n_latent: The number of latent features for truncated SVD.
    :return: _sparse_features, The sparse matrix of item-similarity features.
    """
    _item_svd = TruncatedSVD(n_components=_n_latent)
    _item_features = _item_svd.fit_transform(_matrix.transpose())
    print('Converting to sparse')
    _sparse_features = sparse.csr_matrix(_item_features)
    return _sparse_features


def generate_similarity_matrix(_features, _metric):
    """
    Generates the similarity matrix from either item or user features
    based on the given similarity metric.
    :param _features: The matrix of user or item features.
    :param _metric: A string indicating which similarity metric should be used.
    :return: _similarity_matrix, The final similarity matrix.
    """
    assert _metric in ['cityblock', 'cosine', 'euclidean', 'l1', 'l2', 'manhattan']
    print('Computing similarity')
    _similarity_matrix = pairwise_distances(_features, metric=_metric)
    return _similarity_matrix


def merge_meta(_meta_path, _map_path, _ratings):
    """
    Merges book metadata with ratings.

    :param _meta_path: Path to book metadata csv.
    :param _map_path: Path to book ID mapping.
    :param _ratings: Dataframe of rating interactions.
    :return: _ratings_meta, a dataframe of metadata and ratings and
    _metadata_lookup, dictionary for the UI.
    """
    _meta = pd.read_csv(_meta_path)
    _map = pd.read_csv(_map_path)
    _ratings_map = _ratings.merge(_map, how='left',
                                  left_on='book_id', right_on='book_id_csv')
    _ratings_map = _ratings_map[['user_id', 'book_id_csv', 'is_read',
                                 'rating', 'is_reviewed', 'book_id_y']]
    _ratings_map.columns = ['user_id', 'book_idx', 'is_read',
                            'rating', 'is_reviewed', 'book_id']
    _metadata_lookup = {}
    for _, row in _ratings_map.iterrows():
        _md = _meta[_meta['book_id'] == row['book_id']]
        _metadata_lookup[str(row.book_idx)] = {
            'title': _md['title'].values[0],
            'link': _md['link'].values[0]}
    return _ratings_map, _metadata_lookup

# Problem 1: Running the Recommender Engines

## Build Phase
| Dataset | Build Time | No. of Users | No. of Books | Factors | Sparsity |
| --- | --- | --- | --- | --- | --- |
| 5k | 0.06s | 6 | 7,076 | 4 | 97.07% |
| 8k | 0.08s | 10 | 7,519 | 7 | 97.45% |
| 9k | 0.14s | 14 | 8,463 | 10 | 98.02% |

All three datasets have a high sparsity i.e. most of the users have rated very few books. Build time scales with the dataset size which is expected as the computation (O(N²)) grows as the number of books increases.

## Test Phase

Testing book index #948: *The Ultimate Hitchhiker's Guide to the Galaxy* across all three datasets.

| Dataset | Match 0 | Match 1 | Match 2 | Match 3 | Match 4 |
| --- | --- | --- | --- | --- | --- |
| 5k | Insurgent (Divergent, #2) | 11/22/63 | The Handmaid's Tale | A Game of Thrones | Stardust |
| 8k | Stardust | Insurgent (Divergent, #2) | The Lion, the Witch, and the Wardrobe | Destroy Me | With All My Soul |
| 9k | The Handmaid's Tale | The Lion, the Witch, and the Wardrobe | A Game of Thrones | The Fault in Our Stars | Stardust |

The recommendations seem to follow the themes of the selected book. Sci-fi, fantasy titles dominate across all three datasets. Books like Stardust and Game of thrones appear across multiple datasets, suggesting they might be highly co-rated with the selected book. The variation in the results between the datasets is expected, as a single addition of new user who has rated the selected book can highly affect the results.

In [46]:
# Try different combinations of dataset sample size
CONFIGS = [
    (5000, '5k',  4), #5k dataset
    (8000, '8k',  7), #8k dataset
    (9000, '9k', 10), #9k dataset
]
METRIC = 'cityblock'
p1_results = {}
for NS, FN, FACTORS in CONFIGS:
    try:
        goodreads = pd.read_csv('goodreads_{}.csv'.format(FN))
    except FileNotFoundError:
        read_raw_data(NS, FN)
        goodreads = pd.read_csv('goodreads_{}.csv'.format(FN))

    ratings_meta, metadata_lookup = merge_meta(
    'book_metadata.csv',
    'book_id_map.csv', goodreads)
    
    print('Saving metadata')
    with open('books_metadata_{records}.json'.format(records=FN), 'w', encoding='utf-8') as f:
        json.dump(metadata_lookup, f)
    
    t0 = time.time()
    ratings = build_rating_matrix(ratings_meta)
    item_features = recommend_item_similarity(ratings, FACTORS)
    sim = generate_similarity_matrix(item_features, METRIC)
    # time taken to compute similarity matrix
    elapsed = time.time() - t0
    
    with open('book_similarity_{factors}_{records}_{metric}.pkl'.format(factors=FACTORS,
                                                                    records=FN,
                                                                    metric=METRIC), 'wb') as f:
        pickle.dump(sim, f)

    # Dataset summary
    n_users, n_books = ratings.shape
    sparsity = 1.0 - np.count_nonzero(ratings) / float(n_users * n_books)
    p1_results[NS] = {
        'n_users': n_users,
        'n_books': n_books,
        'factors': FACTORS,
        'sparsity': sparsity,
        'elapsed_time': elapsed
    }
    print('Dataset: {}, Users: {}, Books: {}, Factors: {}, Sparsity: {:.4f}, Elapsed Time: {:.2f} seconds'.format(
        FN, n_users, n_books, FACTORS, sparsity, elapsed))

Saving metadata
Users: 6
Books: 7076


1329it [00:00, 68905.27it/s]

Converting to sparse
Computing similarity


Dataset: 5k, Users: 6, Books: 7076, Factors: 4, Sparsity: 0.9707, Elapsed Time: 0.06 seconds
Saving metadata
Users: 10
Books: 7519


2024it [00:00, 71441.07it/s]

Converting to sparse
Computing similarity


Dataset: 8k, Users: 10, Books: 7519, Factors: 7, Sparsity: 0.9745, Elapsed Time: 0.08 seconds
Saving metadata
Users: 14
Books: 8463


2476it [00:00, 73669.89it/s]

Converting to sparse
Computing similarity


Dataset: 9k, Users: 14, Books: 8463, Factors: 10, Sparsity: 0.9802, Elapsed Time: 0.14 seconds


In [47]:
def test_recommender(_search, _similarity, _metadata):
    """
    A function to test our recommender system.

    :param _search: A book ID to search for.
    :param _similarity: Our recommender similarity matrix.
    :param _metadata: Mapping of book ID to title.
    :return: List of titles of top 5 most similar books.
    """
    row_sims = _similarity[_search, ]
    # Only rank indices that have metadata entries (not all matrix indices are rated books)
    valid_indices = [i for i in range(len(row_sims)) if str(i) in _metadata and i != _search]
    res = sorted(valid_indices, key=lambda sub: row_sims[sub])[-5:]
    res.reverse()
    print('Searched for book: {sb}'.format(sb=_metadata[str(_search)]['title']))
    for j, _ in enumerate(res):
        print('Match {idx}: {book}'.format(idx=j, book=_metadata[str(res[j])]['title']))

In [48]:
# test the query function on all three datasets
for NS, FN, FACTORS in CONFIGS:
    with open('books_metadata_{records}.json'.format(records=FN), 'r', encoding='utf-8') as f:
        metadata_lookup = json.load(f)
    with open('book_similarity_{factors}_{records}_{metric}.pkl'.format(factors=FACTORS,
                                                                    records=FN,
                                                                    metric=METRIC), 'rb') as f:
        sim = pickle.load(f)
    print('\nTesting on dataset: {}'.format(FN))
    test_recommender(948, sim, metadata_lookup)


Testing on dataset: 5k
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: Insurgent (Divergent, #2)
Match 1: 11/22/63
Match 2: The Handmaid's Tale
Match 3: A Game of Thrones (A Song of Ice and Fire, #1)
Match 4: Stardust

Testing on dataset: 8k
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: Stardust
Match 1: Insurgent (Divergent, #2)
Match 2: The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)
Match 3: Destroy Me (Shatter Me, #1.5)
Match 4: With All My Soul (Soul Screamers, #7)

Testing on dataset: 9k
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: The Handmaid's Tale
Match 1: The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)
Match 2: A Game of Thrones (A Song of Ice and Fire, #1)
Match 3: The Fau

# Problem 2: Parameter Tuning

## Number of Latent Factors
|Latent Factors|Top Recommendations|Comments|
| --- | --- | --- |
|1|The Golden Statue Plot, Deluxe Essential Handbook, Prague Fatale, Pride and Prejudice, A Tale of Two Cities|Recommendation seem random with no correlation to the selected book|
|2|Destroy Me, With All My Soul, Throne of Glass, Reaper, Insurgent|Recommendation seem to improve but may be dominated by a single user's pattern|
|4|11/22/63, Misery, Night, The Handmaid's Tale, The Lovely Bones|Recommendations span multiple genres|
|7|The Handmaid's Tale, Insurgent, Night, The Lion the Witch and the Wardrobe, Harry Potter|Recommendation further improve clustering across multiple genres|
|10|The Handmaid's Tale, Narnia, A Game of Thrones, The Fault in Our Stars, Stardust|Strong results with consistent genre coherence across queries|
|13|The Hunger Games, Narnia, The Handmaid's Tale, Catching Fire, Harry Potter|Good results but at the maximum allowed value the model risks overfitting sparse user patterns|

Number of factors are constrained by the number of users since TruncatedSVD decomposes the (books × users) matrix and cannot extract more components than there are columns (users). Valid range: 1-13 for the 9k dataset.

Based on the results, **FACTORS = 7-10** gives the best balance. Below 4 the recommendations are random or genre-locked to one user's taste; at the maximum the model starts memorising individual user patterns.

## Similarity Metric
Results below use FACTORS=10 on the 9k dataset, querying book index #948.

| Metric | Top Recommendations | Comments |
| --- | --- | --- |
| cityblock/l1/manhattan | Handmaid's Tale, Narnia, Game of Thrones, Fault in Our Stars, Stardust | Computes similarity based on the distance, is considered robust to outliers. Works best for grid like data. |
| cosine | Mortal Instruments, Delirium, Lips Touch, The Social Animal, Fault in Our Stars | Measures angle between vectors, ignores magnitude; unstable for low-dimensional data |
| euclidean/l2 | Night, Handmaid's Tale, Insurgent, 1984, Harry Potter | Straight line distance between two points. Is generally considered a good baseline.|

**Best metric: cityblock / l1 / manhattan**. They distribute sensitivity evenly across all latent dimensions, which is important when some SVD dimensions have large variance due to a dominant user. Euclidean/l2 penalises large differences quadratically, making it sensitive to outlier dimensions. Cosine ignores magnitude entirely and produces the most divergent results.

**Degenerate cases:**
- With FACTORS=1 all recommendations collapse onto one axis and are completely unrelated to the query.
- cosine with FACTORS<=2 produces unstable results which suggests that inm the current case of goodreads dataset magnitude matters as cosine ignores magnitude.

**Best overall configuration: FACTORS=10, cityblock, 9k dataset** - consistent genre-coherent results across multiple test queries.

In [56]:
# Try different numbers of latent factors on the 9k dataset
NS, FN = 9000, '9k'
METRIC = 'cityblock'

try:
    goodreads = pd.read_csv('goodreads_{}.csv'.format(FN))
except FileNotFoundError:
    read_raw_data(NS, FN)
    goodreads = pd.read_csv('goodreads_{}.csv'.format(FN))

ratings_meta, metadata_lookup = merge_meta(
    'book_metadata.csv',
    'book_id_map.csv', goodreads)
with open('books_metadata_{records}.json'.format(records=FN), 'w', encoding='utf-8') as f:
    json.dump(metadata_lookup, f)

ratings = build_rating_matrix(ratings_meta)
n_users, n_books = ratings.shape
print('Dataset: {}, Users: {}, Books: {}'.format(FN, n_users, n_books)) 

for FACTORS in [1, 2, 4, 7, 10, 13]:
    print('\nTesting with {} latent factors'.format(FACTORS))
    try:
        t0 = time.time()
        item_features = recommend_item_similarity(ratings, FACTORS)
        sim = generate_similarity_matrix(item_features, METRIC)
        elapsed = time.time() - t0
        with open('book_similarity_{factors}_{records}_{metric}.pkl'.format(factors=FACTORS,
                                                                    records=FN,
                                                                    metric=METRIC), 'wb') as f:
            pickle.dump(sim, f)
        print('Elapsed Time: {:.2f} seconds'.format(elapsed))
        # Test the recommender with the new similarity matrix
        test_recommender(948, sim, metadata_lookup)
    except Exception as e:
        print('Error with {} latent factors: {}'.format(FACTORS, e))

Users: 14
Books: 8463


2476it [00:00, 70398.84it/s]

Dataset: 9k, Users: 14, Books: 8463

Testing with 1 latent factors
Converting to sparse
Computing similarity


Elapsed Time: 0.04 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: The Golden Statue Plot (Geronimo Stilton, #55)
Match 1: Deluxe Essential Handbook
Match 2: Prague Fatale (Bernard Gunther, #8)
Match 3: Pride and Prejudice
Match 4: A Tale of Two Cities

Testing with 2 latent factors
Converting to sparse
Computing similarity
Elapsed Time: 0.05 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: Destroy Me (Shatter Me, #1.5)
Match 1: With All My Soul (Soul Screamers, #7)
Match 2: Throne of Glass (Throne of Glass, #1)
Match 3: Reaper (Soul Screamers, #3.5)
Match 4: Insurgent (Divergent, #2)

Testing with 4 latent factors
Converting to sparse
Computing similarity
Elapsed Time: 0.05 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the G

In [60]:
# try different similarity metrics on the 9k dataset with 10 latent factors
NS, FN, FACTORS = 9000, '9k', 10

features = recommend_item_similarity(ratings, FACTORS)
for METRIC in ['cityblock', 'euclidean', 'cosine']:
    print('\nTesting with {} similarity metric'.format(METRIC))
    try:
        t0 = time.time()
        sim = generate_similarity_matrix(features, METRIC)
        elapsed = time.time() - t0
        with open('book_similarity_{factors}_{records}_{metric}.pkl'.format(factors=FACTORS,
                                                                    records=FN,
                                                                    metric=METRIC), 'wb') as f:
            pickle.dump(sim, f)
        print('Elapsed Time: {:.2f} seconds'.format(elapsed))
        # Test the recommender with the new similarity matrix
        test_recommender(948, sim, metadata_lookup)
    except Exception as e:
        print('Error with {} similarity metric: {}'.format(METRIC, e))

Converting to sparse

Testing with cityblock similarity metric
Computing similarity
Elapsed Time: 0.11 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: The Handmaid's Tale
Match 1: The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)
Match 2: A Game of Thrones (A Song of Ice and Fire, #1)
Match 3: The Fault in Our Stars
Match 4: Stardust

Testing with euclidean similarity metric
Computing similarity
Elapsed Time: 0.34 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: Night (The Night Trilogy #1)
Match 1: The Handmaid's Tale
Match 2: Insurgent (Divergent, #2)
Match 3: 1984
Match 4: Harry Potter and the Sorcerer's Stone (Harry Potter, #1)

Testing with cosine similarity metric
Computing similarity
Elapsed Time: 0.50 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete

In [64]:
# Try different combinations of factors and similarity metrics on the 9k dataset
FACTORS = [1, 2, 4, 7, 10, 13]
METRICS = ['cityblock', 'euclidean', 'cosine']

print(f'searching for best combination of factors and similarity metrics on the 9k dataset')
for FACTORS in FACTORS:
    features = recommend_item_similarity(ratings, FACTORS)
    for METRIC in METRICS:
        print('\nTesting with {} latent factors and {} similarity metric'.format(FACTORS, METRIC))
        try:
            t0 = time.time()
            sim = generate_similarity_matrix(features, METRIC)
            elapsed = time.time() - t0
            with open('book_similarity_{factors}_{records}_{metric}.pkl'.format(factors=FACTORS,
                                                                        records=FN,
                                                                        metric=METRIC), 'wb') as f:
                pickle.dump(sim, f)
            print('Elapsed Time: {:.2f} seconds'.format(elapsed))
            # Test the recommender with the new similarity matrix
            test_recommender(948, sim, metadata_lookup)
        except Exception as e:
            print('Error with {} latent factors and {} similarity metric: {}'.format(FACTORS, METRIC, e))

searching for best combination of factors and similarity metrics on the 9k dataset
Converting to sparse

Testing with 1 latent factors and cityblock similarity metric
Computing similarity
Elapsed Time: 0.04 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: Prague Fatale (Bernard Gunther, #8)
Match 1: Pride and Prejudice
Match 2: A Tale of Two Cities
Match 3: The Lost Art of Reading Nature's Signs: Use Outdoor Clues to Find Your Way, Predict the Weather, Locate Water, Track Animals—and Other Forgotten Skills
Match 4: The Doors of Perception & Heaven and Hell

Testing with 1 latent factors and euclidean similarity metric
Computing similarity
Elapsed Time: 0.25 seconds
Searched for book: The Ultimate Hitchhiker's Guide: Five Complete Novels and One Story (Hitchhiker's Guide to the Galaxy, #1-5)
Match 0: Prague Fatale (Bernard Gunther, #8)
Match 1: Pride and Prejudice
Match 2: A Tale of Two Citie

# Problem 3: User-User Recommendations

## Implementation
The user-user system reuses the same pipeline but applies SVD to the rating matrix directly rather than its transpose:

- **Item-based:** SVD on **(books × users)** → one latent embedding per **book**
- **User-based:** SVD on **(users × books)** → one latent embedding per **user**

Given a user ID, the system returns the 5 most similar users in embedding space, then collects all books those users rated highly that the query user has not yet read.

## Results
On the 9k dataset (FACTORS=10, cosine), top 5 similar users for *user id #0* are identified and their highest-rated unread books are returned. With only 14 users total, all users are necessarily "close" in embedding space — the small user pool is the limiting factor for the user-user model.

## Hyperparameter Tuning for User-User
The constraint for user-user is the inverse of item-based: FACTORS must be less than the number of **books** (columns), which is always in the thousands. This means user-user accepts a much wider range of factors without erroring. However, with only 6-14 users, high factor counts produce overfit embeddings where every user appears maximally different.
We also observe that increasing the number of factors beyond a point does not change the results i.e. adding more factors does not change the embeddings further. Results from number of factors as 1000, or 5000 are identical to the results we had for 13. As the dataset only has 14 users the matrix does not have more non-trivial values.

Best results: **FACTORS=4-7, cosine or cityblock**. At FACTORS=1-2 the user embeddings collapse and all users appear similar regardless of reading taste. At high factors the model memorises each user's exact rating profile, failing to generalise.

## Item-Based vs User-Based: Comparison

| Dimension | Item-Based | User-Based |
| --- | --- | --- |
| What it learns | Books that attract similar audiences | Users with overlapping reading tastes |
| Recommendation Criteria | "If you liked X, try Y" | "People like you also read Z" |
| Scalability | Similarity computed once; stable as long as catalogue is fixed | Must be recomputed as users add or change ratings |
| Cold start — new book | Cannot recommend until the book has at least one rating | Unaffected |
| Cold start — new user | Can recommend from any seed book without rating history | Cannot find neighbours without rating history |
| SVD constraint | FACTORS < n_users (6-14) | FACTORS < n_books  (thousands) |
| Suitability for Goodreads dataset | Better — many books, very few users | Weaker — too few users to form a meaningful pattern |

**Summary:** Item-based similarity is better suited to this dataset. With only 6-14 users, user-user similarity factor is severely limited. Model does not have enough users to find a meaningful pattern, and the SVD factor constraint becomes a practical bottleneck. In a scenario where the dataset has more number of users (1K~1M), user-based collaborative filtering would be the stronger approach, enabling personalised discovery beyond genre. Item-based is preferred when the catalogue is large and stable, or when recommendations need to work for anonymous visitors without rating history.

In [70]:
# User-User based recommendation system
def recommend_user_similarity(_matrix, _n_latent):
    """
    Builds user similarities using truncated SVD.
    :param _matrix: The user-item rating matrix.
    :param _n_latent: The number of latent features for truncated SVD.
    :return: _sparse_features, The sparse matrix of user-similarity features.
    """
    _user_svd = TruncatedSVD(n_components=_n_latent)
    _user_features = _user_svd.fit_transform(_matrix)
    print('Converting to sparse')
    _sparse_features = sparse.csr_matrix(_user_features)
    return _sparse_features

def test_user_recommender(_search, _similarity, _ratings_matrix, _metadata):
    """
    A function to test our user-based recommender system.

    :param _search: A user ID to search for.
    :param _similarity: Our recommender similarity matrix.
    :param _ratings_matrix: The per-dataset ratings matrix (users x books numpy array).
    :param _metadata: Mapping of book ID to title.
    :return: List of titles of top 5 most similar books.
    """
    row_sims = _similarity[_search, ]
    res = sorted(range(len(row_sims)), key=lambda sub: row_sims[sub])[-5:]
    res.reverse()
    print('Searched for user: {sb}'.format(sb=_search))
    for j, _ in enumerate(res):
        print('Match {idx}: User {user}'.format(idx=j, user=res[j]))

    already_rated = np.where(_ratings_matrix[_search, ] > 0)[0]
    # number of books already rated by the user
    print('Books already rated by user {}: {}'.format(_search, len(already_rated)))

    # Collect top rated books from similar users
    recommended_books = {}
    for similar_user in res:
        similar_user_ratings = _ratings_matrix[similar_user, ]
        for book_idx in np.where(similar_user_ratings > 0)[0]:
            # Guard: only look up indices present in this dataset's metadata
            if book_idx not in already_rated and str(book_idx) in _metadata:
                if book_idx not in recommended_books:
                    recommended_books[book_idx] = []
                recommended_books[book_idx].append(similar_user_ratings[book_idx])
    # Sort recommended books by average rating from similar users
    sorted_recommendations = sorted(recommended_books.items(), key=lambda x: np.mean(x[1]), reverse=True)[:5]
    print('Top recommended books for user {}:'.format(_search))
    for book_idx, ratings_list in sorted_recommendations:
        print('Book: {}, Average Rating from Similar Users: {:.2f}'.format(
            _metadata[str(book_idx)]['title'], np.mean(ratings_list)))

In [71]:
# Build user similarity matrix and test the user-based recommender
U_FACTOR = 10
U_METRIC = 'cosine'

U_sims, U_ratings, U_metas = {}, {}, {}

for NS, FN, _ in CONFIGS:
    print("User-User based recommendation for dataset: {}".format(FN))
    try:
        goodreads = pd.read_csv('goodreads_{}.csv'.format(FN))
    except FileNotFoundError:
        read_raw_data(NS, FN)
        goodreads = pd.read_csv('goodreads_{}.csv'.format(FN))
    
    ratings_meta, metadata_lookup = merge_meta(
        'book_metadata.csv',
        'book_id_map.csv', goodreads)
    t0 = time.time()
    ratings = build_rating_matrix(ratings_meta)
    user_features = recommend_user_similarity(ratings, U_FACTOR)
    sim = generate_similarity_matrix(user_features, U_METRIC)
    elapsed = time.time() - t0
    print('Elapsed Time: {:.2f} seconds'.format(elapsed))

    U_sims[FN] = sim
    U_ratings[FN] = ratings
    U_metas[FN] = metadata_lookup

User-User based recommendation for dataset: 5k
Users: 6
Books: 7076


1329it [00:00, 65121.03it/s]

Converting to sparse
Computing similarity
Elapsed Time: 0.03 seconds
User-User based recommendation for dataset: 8k


Users: 10
Books: 7519


2024it [00:00, 62459.23it/s]

Converting to sparse
Computing similarity
Elapsed Time: 0.04 seconds
User-User based recommendation for dataset: 9k


Users: 14
Books: 8463


2476it [00:00, 67549.30it/s]

Converting to sparse
Computing similarity
Elapsed Time: 0.05 seconds


In [72]:
# Test the user-based recommender on all three datasets
for NS, FN, _ in CONFIGS:
    print('\nTesting user-based recommender on dataset: {}'.format(FN))
    test_user_recommender(0, U_sims[FN], U_ratings[FN], U_metas[FN])


Testing user-based recommender on dataset: 5k
Searched for user: 0
Match 0: User 5
Match 1: User 3
Match 2: User 4
Match 3: User 2
Match 4: User 1
Books already rated by user 0: 473
Top recommended books for user 0:
Book: Leopard Dreaming (Mira Chambers #3), Average Rating from Similar Users: 5.00
Book: Hindsight (Mira Chambers #2), Average Rating from Similar Users: 5.00
Book: The Transfer (Divergent, #0.1), Average Rating from Similar Users: 5.00
Book: Before I Fall, Average Rating from Similar Users: 5.00
Book: All the Snake Handlers I Know Are Dead, Average Rating from Similar Users: 5.00

Testing user-based recommender on dataset: 8k
Searched for user: 0
Match 0: User 5
Match 1: User 9
Match 2: User 3
Match 3: User 4
Match 4: User 7
Books already rated by user 0: 473
Top recommended books for user 0:
Book: Thirteen Reasons Why, Average Rating from Similar Users: 5.00
Book: Pretty Baby, Average Rating from Similar Users: 5.00
Book: The Wishing Well (The Lone City, #0.2), Average R

In [73]:
# Try different factors and similarity metrics for the user-based recommender on the 9k dataset
NS, FN = 9000, '9k'
for FACTORS in [1, 2, 4, 7, 10, 13, 1000, 5000]:
    for METRIC in ['cityblock', 'euclidean', 'cosine']:
        print('\nTesting user-based recommender with {} latent factors and {} similarity metric on dataset: {}'.format(FACTORS, METRIC, FN))
        try:
            user_features = recommend_user_similarity(ratings, FACTORS)
            sim = generate_similarity_matrix(user_features, METRIC)
            test_user_recommender(0, sim, ratings, metadata_lookup)
        except Exception as e:
            print('Error with {} latent factors and {} similarity metric: {}'.format(FACTORS, METRIC, e))


Testing user-based recommender with 1 latent factors and cityblock similarity metric on dataset: 9k
Converting to sparse
Computing similarity
Searched for user: 0
Match 0: User 10
Match 1: User 5
Match 2: User 3
Match 3: User 9
Match 4: User 7
Books already rated by user 0: 473
Top recommended books for user 0:
Book: Deluxe Essential Handbook, Average Rating from Similar Users: 5.00
Book: The Golden Statue Plot (Geronimo Stilton, #55), Average Rating from Similar Users: 5.00
Book: 11/22/63, Average Rating from Similar Users: 5.00
Book: We Were Liars, Average Rating from Similar Users: 5.00
Book: Thirteen Reasons Why, Average Rating from Similar Users: 5.00

Testing user-based recommender with 1 latent factors and euclidean similarity metric on dataset: 9k
Converting to sparse
Computing similarity
Searched for user: 0
Match 0: User 10
Match 1: User 5
Match 2: User 3
Match 3: User 9
Match 4: User 7
Books already rated by user 0: 473
Top recommended books for user 0:
Book: Deluxe Essenti